<a href="https://colab.research.google.com/github/AhmadKevin000/machineLearning/blob/main/week11_tugasPrak.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Praktikum — SVM


## Soal 1 — Klasifikasi Data `voice.csv`

In [71]:
# import library
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

**Soal:** Load data `voice.csv`, pisahkan fitur dan label, lalu lakukan encoding label.

In [72]:
# Load data
df = pd.read_csv('voice.csv')
df.head()

,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [73]:
# Cek informasi data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3168 entries, 0 to 3167
Data columns (total 21 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   meanfreq  3168 non-null   float64
 1   sd        3168 non-null   float64
 2   median    3168 non-null   float64
 3   Q25       3168 non-null   float64
 4   Q75       3168 non-null   float64
 5   IQR       3168 non-null   float64
 6   skew      3168 non-null   float64
 7   kurt      3168 non-null   float64
 8   sp.ent    3168 non-null   float64
 9   sfm       3168 non-null   float64
 10  mode      3168 non-null   float64
 11  centroid  3168 non-null   float64
 12  meanfun   3168 non-null   float64
 13  minfun    3168 non-null   float64
 14  maxfun    3168 non-null   float64
 15  meandom   3168 non-null   float64
 16  mindom    3168 non-null   float64
 17  maxdom    3168 non-null   float64
 18  dfrange   3168 non-null   float64
 19  modindx   3168 non-null   float64
 20  label     3168 non-null   obje

In [74]:
# Cek distribusi label
df['label'].value_counts()

,count
label,
male,1584
female,1584


In [75]:
# Pisahkan fitur dan label, lalu encode label
X = df.drop('label', axis=1)
y = df['label'].astype('category').cat.codes  # male/female -> 0/1

print(X.shape, y.shape)

(3168, 20) (3168,)


**Soal:** Split data dengan rasio **70:30** dan **80:20**, lalu bangun model SVM dengan kernel **linier**, **polynomial**, dan **RBF** untuk setiap split. Standardisasi fitur dilakukan di dalam pipeline agar tidak terjadi data leakage.

In [76]:
# Split data 70:30 dan 80:20
X_train70, X_test70, y_train70, y_test70 = train_test_split(X, y, test_size=0.3, random_state=42)
X_train80, X_test80, y_train80, y_test80 = train_test_split(X, y, test_size=0.2, random_state=42)

In [77]:
# Definisikan kernel yang diuji
kernels = {'linier': 'linear', 'polynomial': 'poly', 'rbf': 'rbf'}

def build_and_eval(Xtr, Xte, ytr, yte):
    results = {}
    for name, kernel in kernels.items():
        model = make_pipeline(StandardScaler(), SVC(kernel=kernel))
        model.fit(Xtr, ytr)
        y_pred = model.predict(Xte)
        results[name] = accuracy_score(yte, y_pred)
    return results

res_70 = build_and_eval(X_train70, X_test70, y_train70, y_test70)
res_80 = build_and_eval(X_train80, X_test80, y_train80, y_test80)

**Soal:** Tabulasikan performansi setiap split dan kernel berdasarkan metrik akurasi.

In [78]:
# Tabulasi hasil akurasi
tabel = pd.DataFrame({
    'Kernel': list(kernels.keys()),
    'Split 70:30': [res_70[k] for k in kernels],
    'Split 80:20': [res_80[k] for k in kernels]
})
tabel

,Kernel,Split 70:30,Split 80:20
0,linier,0.970557,0.976341
1,polynomial,0.957939,0.971609
2,rbf,0.981073,0.982650


**Analisis:** *(isi analisis Anda di sini — misalnya kernel mana yang paling baik, dan apakah rasio split berpengaruh terhadap akurasi)*

## Soal 2 — Klasifikasi Siang/Malam dengan Fitur Histogram (SVM Kernel RBF)

Menggunakan data Praktikum 5. Fitur yang digunakan adalah **histogram** dari citra, dengan rasio split **80:20** dan hyperparameter tunning pada kernel RBF.

### Pra Pengolahan Data (dari Praktikum 5)

In [79]:
# import library
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import cv2
import numpy as np
import pandas as pd

In [80]:
# Image directories
train_dir = "images/training/"
test_dir = "images/test/"

In [81]:
# Load dataset (dari Praktikum 5)
def load_dataset(img_dir):
    p = Path(img_dir)
    dirs = p.glob('*')
    img_list = []
    for dir in dirs:
        label = str(dir).split('/')[-1]
        for file in dir.glob('*.jpg'):
            img = mpimg.imread(file)
            if not img is None:
                img_list.append((img, label))
    return img_list

def standarized_input(image):
    # resize to w: 1100, h:600
    std_img = cv2.resize(image, (1100, 600))
    return std_img

def label_encoder(label):
    # day as 1; night as 0
    return 1 if label == 'day' else 0

def preprocess(img_list):
    std_img_list = []
    for item in img_list:
        std_img = standarized_input(item[0])
        img_label = label_encoder(item[1])
        std_img_list.append((std_img, img_label))
    return std_img_list

In [82]:
# Load dan preprocess seluruh data (training + test digabung, lalu di-split 80:20)
train_img = load_dataset(train_dir)
test_img = load_dataset(test_dir)

all_img = preprocess(train_img) + preprocess(test_img)
print(f'Total data: {len(all_img)}')

Total data: 400


### Ekstraksi Fitur Histogram

**Soal:** Buat fungsi ekstraksi fitur **histogram** dari citra, lalu tabulasi fitur dan label dalam dataframe.

In [83]:
# Ekstraksi fitur histogram
def histogram_feature(image, bins=64):
    # Konversi ke grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    # Hitung histogram dengan normalisasi (agar tidak tergantung jumlah piksel)
    hist = cv2.calcHist([gray], [0], None, [bins], [0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    return hist

In [84]:
# Bangun feature vectors dari histogram
features = []
labels = []
for img, label in all_img:
    features.append(histogram_feature(img))
    labels.append(label)

X_img = np.array(features)
y_img = np.array(labels)
print(f'Feature shape: {X_img.shape}')

Feature shape: (400, 64)


### Split Data 80:20 dan Hyperparameter Tunning

**Soal:** Split data dengan rasio 80:20, lalu lakukan hyperparameter tunning kernel RBF menggunakan GridSearch untuk nilai C dan gamma.

In [85]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Split 80:20
X_train, X_test, y_train, y_test = train_test_split(X_img, y_img, test_size=0.2,
                                                    random_state=42, stratify=y_img)

In [86]:
import os
print("Current working directory content:", os.listdir('.'))
if os.path.exists('images'):
    print("\n'images' directory content:", os.listdir('images'))
else:
    print("\n'images' directory does not exist in the current path.")

Current working directory content: ['.config', 'images.zip', 'images', '__MACOSX', 'voice.csv', 'sample_data']

'images' directory content: ['training', 'test']


In [87]:
import os

paths = ['images', 'images/training', 'images/test']
for path in paths:
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"Folder '{path}' terdeteksi! Berisi {len(files)} file/direktori.")
        if len(files) > 0:
            print(f"  Contoh isi: {files[:5]}")
    else:
        print(f"Folder '{path}' TIDAK ditemukan.")

Folder 'images' terdeteksi! Berisi 2 file/direktori.
  Contoh isi: ['training', 'test']
Folder 'images/training' terdeteksi! Berisi 2 file/direktori.
  Contoh isi: ['day', 'night']
Folder 'images/test' terdeteksi! Berisi 2 file/direktori.
  Contoh isi: ['day', 'night']


In [88]:
import zipfile
import os

zip_path = 'images.zip'
extract_path = '.'

if os.path.exists(zip_path):
    print(f"Mengekstrak {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Ekstraksi selesai!")
else:
    print(f"File {zip_path} tidak ditemukan di direktori aktif.")

Mengekstrak images.zip...
Ekstraksi selesai!


In [89]:
paths = ['images', 'images/training', 'images/test']
for path in paths:
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"Folder '{path}' terdeteksi! Berisi {len(files)} file/direktori.")
        if len(files) > 0:
            print(f"  Contoh isi: {files[:5]}")
    else:
        print(f"Folder '{path}' TIDAK ditemukan.")

Folder 'images' terdeteksi! Berisi 2 file/direktori.
  Contoh isi: ['training', 'test']
Folder 'images/training' terdeteksi! Berisi 2 file/direktori.
  Contoh isi: ['day', 'night']
Folder 'images/test' terdeteksi! Berisi 2 file/direktori.
  Contoh isi: ['day', 'night']


In [90]:
train_img = load_dataset(train_dir)
test_img = load_dataset(test_dir)

all_img = preprocess(train_img) + preprocess(test_img)
print(f'Total data gambar terproses: {len(all_img)}')

Total data gambar terproses: 400


In [91]:
# Membangun ulang feature vectors
features = []
labels = []
for img, label in all_img:
    features.append(histogram_feature(img))
    labels.append(label)

X_img = np.array(features)
y_img = np.array(labels)
print(f'Feature shape baru: {X_img.shape}')

Feature shape baru: (400, 64)


In [92]:
# Hyperparameter tunning kernel RBF
param_grid = {'svc__C': [1, 5, 10, 50, 100],
              'svc__gamma': [0.0001, 0.0005, 0.001, 0.005, 'scale']}

grid = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel='rbf')),
                    param_grid, cv=5)
%time grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)

CPU times: user 780 ms, sys: 2.69 ms, total: 783 ms
Wall time: 791 ms
{'svc__C': 10, 'svc__gamma': 0.005}
0.996875


**Soal:** Gunakan model terbaik untuk memprediksi data testing dan catat performansi akurasinya.

In [93]:
# Evaluasi model terbaik
best_model = grid.best_estimator_

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

acc_train = accuracy_score(y_train, y_train_pred)
acc_test = accuracy_score(y_test, y_test_pred)

print(f'Accuracy on train: {acc_train:.4f}')
print(f'Accuracy on test: {acc_test:.4f}')

Accuracy on train: 0.9969
Accuracy on test: 0.9875


In [94]:
# Detail performansi
from sklearn.metrics import classification_report
print(classification_report(y_test, y_test_pred, target_names=['night', 'day']))

              precision    recall  f1-score   support

       night       0.98      1.00      0.99        40
         day       1.00      0.97      0.99        40

    accuracy                           0.99        80
   macro avg       0.99      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80



**Analisis:** *(isi catatan performansi Anda di sini — bandingkan akurasi fitur histogram dengan fitur average brightness pada Praktikum 5, serta pengaruh nilai C dan gamma hasil tunning)*